## Resume Analyzer using DeepSeek-R1 API


- DeepSeek-R1-0528 is a May 2025 upgrade to DeepSeek's flagship open-source reasoning model (671B total parameters, ~37B active via MoE).

- Built on the V3 base, it delivers dramatically deeper chain-of-thought reasoning—often using 20K+ thinking tokens—pushing performance close to or matching top proprietary models like OpenAI o1/o3, Gemini 2.5 Pro, and Claude 4 in math (e.g. 87.5% AIME), coding, complex logic, and tool use.

- Key gains include fewer hallucinations, native JSON/function calling, and stronger agentic behavior.

- Fully MIT-licensed weights are available on Hugging Face; distilled 8B variants bring near-frontier quality to smaller hardware.

In [1]:
from openai import OpenAI
from rich.console import Console
from rich.markdown import Markdown
console = Console()

from google.colab import userdata
API_KEY = userdata.get('DeepSeekR1')

from pypdf import PdfReader
client = OpenAI(base_url = "https://openrouter.ai/api/v1", api_key=API_KEY)

In [2]:
# This function will extract the contents from the pdf files
def extract_content(pdf_path):
  doc = PdfReader(pdf_path)
  text = ""
  for page in doc.pages:
    text += page.extract_text()
  return text

In [3]:
pdf_path = "/content/resume_analyzer_deepseek/Hrishikesh_Dhole_Resume_DA.pdf"
pdf_content = extract_content(pdf_path)

In [4]:
# The function will analyze the contents of the resume with job description using the model
def analyze_resume(resume_content, job_description):

  prompt = f"""
          You are an ATS system + senior hiring manager for Data Analyst roles.

          Your job is to analyze the resume against the job description using both:
          1. ATS keyword matching logic
          2. Human recruiter 7-second scan evaluation

          Resume:
          {resume_content}

          Job Description:
          {job_description}

          Evaluation Instructions:

          1. Extract all required hard skills from the Job Description.
          2. Extract all preferred skills.
          3. Extract soft skills and business skills.
          4. Compare them to the resume content.
          5. Calculate a weighted match score:
            - 40% Hard Skills Match
            - 20% Tools/Technologies Match
            - 15% Experience Alignment
            - 15% Keyword Density
            - 10% Formatting & Clarity

          6. Identify:
            - Missing hard skills
            - Missing tools/platforms
            - Missing business/analytical phrases
            - Underrepresented keywords
            - Title misalignment (if any)

          7. Evaluate resume quality:
            - Bullet clarity (strong/weak)
            - Quantified metrics presence
            - Action verbs usage
            - ATS formatting risks
            - Keyword stuffing risks

          8. Suggest:
            - Exact bullet rewrites (max 5)
            - Skills section rewrite
            - Summary rewrite aligned to JD
            - Title adjustments if needed

          Return output in this structured format:

          =============================
          ATS MATCH REPORT
          =============================

          Overall Match Score: XX/100

          Hard Skills Match: XX%
          Tools & Platforms Match: XX%
          Experience Alignment: XX%
          Keyword Density Score: XX%
          Formatting Score: XX%

          Missing Hard Skills:
          - ...

          Missing Tools/Platforms:
          - ...

          Missing Business Keywords:
          - ...

          Underrepresented Keywords:
          - ...

          Title Alignment Issues:
          - ...

          Bullet Improvements:
          1. Original: ...
            Improved: ...

          2. Original: ...
            Improved: ...

          Skills Section Optimization:
          - ...

          Summary Optimization:
          - ...

          Final Recommendation:
          (Should candidate apply? Yes/No + reasoning)
          """

  chat = client.chat.completions.create(
    model="deepseek/deepseek-r1-0528:free",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
  )

  return chat.choices[0].message.content

In [5]:
# Paste the job description here
description =  """
About the job
Join a newly created team dedicated to the Disney+ Subscriber Perks program, where you'll transform complex data into strategic business decisions that shape the future of our perks program. Collaborating closely with cross-functional partners in Commerce, Product, Marketing, and Engineering, you'll architect and execute sophisticated experiments that optimize every aspect of our perks program—from initial acquisition through long-term engagement with subscriber perks.

As part of Disney's rapidly evolving streaming ecosystem, you'll tackle complex business challenges that directly impact millions of subscribers across Disney+, Hulu, and ESPN+. Your insights will shape product roadmaps, user experience optimizations, and partnership decisions that drive measurable business growth.

What You'll Do

Design and Execute Experiments: Lead end-to-end A/B testing initiatives and Geo Experiments, from hypothesis formation and experimental design to statistical analysis and business recommendations
Apply Causal Inference Methods: Leverage advanced techniques including difference-in-differences, instrumental variables, and quasi-experimental designs to extract actionable insights from observational data
Drive Strategic Insights: Partner with Product, Marketing, Partnerships, and Finance teams to identify optimization opportunities and translate complex analytical findings into clear business recommendations
Influence Executive Decisions: Present findings and recommendations to senior leadership, effectively communicating statistical concepts to non-technical stakeholders
Build Scalable Solutions: Develop automated experimentation pipelines, causal inference framework, and user-facing tools that can scale across marketing platforms

Required Qualifications & Skills

Strong programming skills in R, Python, or similar languages.
Expertise in using SQL to explore and analyze large data sets.
Familiarity with data platforms and applications such as Databricks, Jupyter, Snowflake, Redshift, Airflow.
Experience with data visualization tools (e.g. Tableau, Power BI, Looker, Plotly, etc.).
Demonstrates strong strategic and analytical thinking, with the ability to interpret market and consumer data, translate complex analyses into clear, actionable insights, and effectively communicate opportunities and challenges to diverse stakeholders.
Familiarity with experimentation design and analysis.
Proven ability to manage end-to-end causal inference analyses, from initial requirements to impactful outcomes.
Practical experience and expertise in applied statistics and data science methodologies (general advanced data modeling and predictive analytics e.g., multivariate regression analysis, machine learning, deep learning, forecasting, Bayesian estimation, causal inference).
Strong ability to build relationships with teammates, business partners, and technical colleagues.
Familiarity with data platforms and applications such as Databricks, Jupyter, Snowflake, Redshift, Airflow.

Preferred Qualifications

Exposure to version control tools like Git.
Experience in the streaming media industry and/or supporting a direct-to-consumer subscription-based product.
Experience with distributed databases and query languages like Spark and Scala.
Experience with Streamlit or React web development.
Familiarity with using or building Gen AI solutions for productivity and efficiency gains.

Required Education

Bachelor's degree in Data Analytics, Computer Science, Data Engineering, Mathematics, Statistics, or comparable field of study, and/or equivalent work experience.

#DISNEYTECH

#DisneyAnalytics

The hiring range for this position in New York, NY is $84,500 to $113,300.00 and in Santa Monica, CA and Glendale, CA is $80,700 to $108,100. The base pay actually offered will take into account internal equity and also may vary depending on the candidate’s geographic region, job-related knowledge, skills, and experience among other factors. A bonus and/or long-term incentive units may be provided as part of the compensation package, in addition to the full range of medical, financial, and/or other benefits, dependent on the level and position offered.
"""

#desc_path = ""
#description = extract_content(desc_path)

In [7]:
# Executeing the analyzer
result = analyze_resume(pdf_content, description)

In [8]:
# Result
print(result)